# S23 · Desarrollo de aplicaciones financieras con Streamlit
**Lenguajes de Programación · Ingeniería Financiera · Taller 6**

**Pregunta:** ¿Cómo construir una aplicación para comparar rendimiento, riesgo y evolución de activos financieros?

**Producto:** Observatorio financiero con datos de yfinance, construido desde Google Colab.

**Antes de programar:** abre la guía HTML de S23 y explora la maqueta del observatorio. Este notebook contiene las explicaciones y el desarrollo práctico.

## ¿Qué vas a construir?
La aplicación tendrá **controles** para seleccionar activos y período; **indicadores** de rendimiento y volatilidad; **gráficos** de precios normalizados, evolución temporal y correlación; y una **lectura financiera** de los resultados.

**Idea fundamental:** un notebook sirve para desarrollar y probar análisis; Streamlit crea una interfaz que otros pueden usar sin modificar el código. GitHub guarda el proyecto, pero no mantiene ejecutándose el servidor.

## 1 · Comprender antes de programar
### ¿Qué es una aplicación web?
Es un programa que se utiliza mediante un navegador. Tiene una **interfaz** (lo que ve y utiliza la persona), una **lógica** (instrucciones que procesan información) y **datos** (información que se consulta y transforma).

**Ejemplo financiero:** el usuario elige *JPMorgan* y *6 meses* → Python consulta precios → Pandas calcula rendimientos → Streamlit muestra métricas y gráficos.

### ¿Qué es Streamlit?
Es un framework de Python para construir aplicaciones de datos mediante instrucciones Python. Permite crear títulos, controles, tablas, gráficos y diseños sin escribir HTML o JavaScript para este laboratorio.

### ¿En qué se diferencia de Colab?
- **Colab:** espacio para escribir y ejecutar celdas, observar resultados y documentar el análisis.
- **Streamlit:** aplicación web que organiza el análisis en una interfaz que responde a selecciones del usuario.
- **Servidor:** proceso que ejecuta la aplicación y entrega su interfaz al navegador.
- **Túnel temporal:** puente externo para abrir desde el navegador el servidor que está dentro de Colab.

**Idea esencial:** cada vez que el usuario cambia un widget, Streamlit normalmente vuelve a ejecutar el script de arriba hacia abajo. Por eso los controles determinan qué datos y resultados se presentan.

**Comprueba tu comprensión:** ¿qué parte de la aplicación selecciona el período?, ¿qué parte calcula el rendimiento?, ¿qué parte lo muestra?

## 2 · Preparar el entorno de trabajo
Una **biblioteca** reúne herramientas ya programadas. `streamlit` construye la interfaz; `yfinance` consulta precios; `pandas` organiza tablas; `numpy` apoya los cálculos.

`pip` instala paquetes en el entorno temporal de Colab. El signo `!` indica que ejecutamos un comando del sistema, no una instrucción Python habitual.

**Ejecuta:**

In [1]:
!pip -q install streamlit yfinance pandas numpy

## 3 · Crear la interfaz
### Conceptos y sintaxis
- `import streamlit as st`: importa Streamlit y permite llamarlo `st`.
- `st.set_page_config(...)`: define el título y la distribución de la página. Debe ir antes de otros comandos `st`.
- `st.title("...")`: muestra el título principal.
- `st.write("...")`: muestra texto u otros objetos.
- `st.info("...")`: presenta una indicación visual.

`%%writefile app.py` es una instrucción especial de Colab que **guarda el contenido de la celda en un archivo**. Ese archivo será nuestra aplicación. No se escribe dentro del archivo Python.

**Observa:** después de ejecutar esta celda, encontrarás `app.py` en el panel de archivos de Colab.

In [2]:
%%writefile app.py
import streamlit as st

st.set_page_config(page_title="Observatorio financiero", layout="wide")
st.title("Observatorio de portafolios")
st.write("Explora acciones y analiza su comportamiento histórico.")
st.info("Seleccionaremos activos, calcularemos indicadores y estudiaremos sus gráficos.")

Writing app.py


## 4 · Ejecutar y abrir la aplicación
`streamlit run app.py` inicia un servidor. En Colab el servidor corre en una máquina remota, así que utilizaremos un enlace temporal para acceder desde el navegador.

**Primero:** inicia el servidor. Si modificas `app.py`, Streamlit normalmente detectará el cambio y recargará la aplicación. Si no se actualiza, recarga la página. No necesitas volver a abrir un túnel por cada cambio.

**Importante:** la conexión puede interrumpirse al terminar la sesión de Colab. No compartas datos privados ni credenciales por el enlace.

In [21]:
import subprocess, time
try:
    servidor.terminate()
except NameError:
    pass
servidor = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port=8501", "--server.headless=true", "--server.address=0.0.0.0"],
    stdout=open("streamlit.log", "w"), stderr=subprocess.STDOUT
)
time.sleep(5)
print("Servidor activo:", servidor.poll() is None)
print("Si aparece False, revisa streamlit.log")

Servidor activo: True
Si aparece False, revisa streamlit.log


**Después:** abre un enlace de acceso temporal. LocalTunnel es un servicio externo que puede fallar o pedir verificación. Mantén esta celda ejecutándose mientras trabajas. Si el enlace no funciona, comprueba primero el servidor con `!tail -n 30 streamlit.log` y solicita la alternativa de acceso indicada por la docente. No introduzcas contraseñas personales en páginas de túneles.

In [ ]:
!npx --yes localtunnel --port 8501

⠙⠹⠸⠼⠴⠦your url is: https://two-cases-sing.loca.lt


## 5 · Darle opciones al usuario
### ¿Qué es un widget?
Un **widget** es un control visual que recibe información del usuario. `st.multiselect()` permite elegir varios activos; `st.selectbox()` permite elegir una opción.

Un **diccionario** relaciona nombres con tickers, por ejemplo `"Apple": "AAPL"`. Una **lista** agrupa elementos y permite recorrerlos. Los estudiantes ya conocen estas estructuras: ahora las utilizarán para construir controles.

**Sintaxis:** `seleccion = st.multiselect("Etiqueta", opciones, default=[...])`.

**Actividad:** ejecuta la celda, vuelve a la aplicación y cambia la selección. Explica por qué la lista de activos seleccionados cambia sin editar `app.py`.

In [ ]:
%%writefile app.py
import streamlit as st

st.set_page_config(page_title="Observatorio financiero", layout="wide")
st.title("Observatorio de portafolios")
st.write("Selecciona activos para comparar su comportamiento.")

activos = {"Apple": "AAPL", "Microsoft": "MSFT", "JPMorgan": "JPM", "NVIDIA": "NVDA", "Alphabet": "GOOGL"}
empresas = st.multiselect("Empresas", list(activos.keys()), default=["Apple", "Microsoft"])
periodo = st.selectbox("Horizonte de análisis", ["3mo", "6mo", "1y", "2y"], index=2)

st.write("Empresas seleccionadas:", empresas)
st.write("Período seleccionado:", periodo)
if not empresas:
    st.warning("Selecciona al menos una empresa para continuar.")

## 6 · Consultar precios históricos
### ¿Qué son yfinance y un DataFrame?
`yfinance` consulta datos de mercado disponibles a través de Yahoo Finance. Un **ticker** es el identificador bursátil de un activo, como `MSFT`. Un **DataFrame** de Pandas es una tabla de filas y columnas; `Close` es el precio de cierre reportado para una sesión.

`yf.Ticker(ticker).history(period=periodo, auto_adjust=True)` solicita precios históricos ajustados para el período elegido. Los datos pueden tener retrasos, faltantes o cambios de disponibilidad.

**Reutilizamos lo aprendido:** un `for` recorre las empresas seleccionadas, un `if` evita procesar tablas vacías y un diccionario guarda las series obtenidas.

**Prueba:** selecciona una empresa y luego dos. Observa cómo cambia la tabla.

In [13]:
%%writefile app.py
import streamlit as st
import yfinance as yf
import pandas as pd

st.set_page_config(page_title="Observatorio financiero", layout="wide")
st.title("Observatorio de portafolios")
activos = {"Apple": "AAPL", "Microsoft": "MSFT", "JPMorgan": "JPM", "NVIDIA": "NVDA", "Alphabet": "GOOGL"}
empresas = st.multiselect("Empresas", list(activos), default=["Apple", "Microsoft"])
periodo = st.selectbox("Período", ["3mo", "6mo", "1y", "2y"], index=2)

series = {}
for empresa in empresas:
    try:
        historial = yf.Ticker(activos[empresa]).history(period=periodo, auto_adjust=True)
        if not historial.empty:
            series[empresa] = historial["Close"].rename(empresa)
    except Exception:
        st.warning(f"No se pudieron consultar los datos de {empresa}.")

if series:
    precios = pd.concat(series.values(), axis=1).sort_index()
    st.subheader("Precios históricos de cierre (USD)")
    st.dataframe(precios.tail(12))
else:
    st.info("Selecciona activos y comprueba la disponibilidad de datos.")

Overwriting app.py


## 7 · Medir el rendimiento
### ¿Qué significa que una acción sea rentable en un período?
El **rendimiento diario simple** mide la variación relativa entre dos cierres consecutivos:

\[r_t=\frac{P_t}{P_{t-1}}-1\]

El **rendimiento acumulado** entre el primer y último cierre disponible es:

\[R=\left(rac{P_{final}}{P_{inicial}}-1
ight)	imes100\%\]

Ejemplo: si el precio pasa de 100 a 110 USD, el rendimiento es **10 %**, no 10 puntos de precio.

En Pandas, `pct_change()` calcula rendimientos entre observaciones consecutivas. `iloc[0]` obtiene el primer valor y `iloc[-1]` el último. **No confundas el rendimiento acumulado con el promedio de rendimientos diarios.**

**Ejercicio:** identifica el activo de mayor rendimiento en el período y explica si el resultado sería necesariamente igual al cambiar el horizonte.

## 8 · Comprender el riesgo y las series temporales
### Volatilidad
La **desviación estándar de los rendimientos diarios** resume cuánto fluctúan alrededor de su media. Para una aproximación anualizada en acciones con unas 252 sesiones bursátiles al año:

\[\sigma_{anual}\approx \sigma_{diaria}\sqrt{252}\]

Una volatilidad mayor indica mayor dispersión histórica de rendimientos; **no significa automáticamente una peor inversión ni predice pérdidas futuras**.

### Medias móviles
Una **media móvil** suaviza el precio y ayuda a observar la tendencia. `rolling(20).mean()` calcula el promedio de las últimas 20 observaciones; `rolling(50).mean()` usa 50. Al inicio de la serie pueden aparecer valores faltantes porque aún no existe una ventana completa.

### Correlación
La correlación entre rendimientos de dos activos varía entre −1 y +1. Un valor positivo alto indica que sus movimientos históricos han tendido a coincidir. **Correlación no implica causalidad** ni garantiza relaciones futuras.

**Pregunta:** ¿por qué conviene comparar rendimientos, y no solamente precios, cuando los activos tienen valores unitarios diferentes?

## 9 · Construir el observatorio completo
Ahora reuniremos controles, consulta de datos, cálculos, indicadores y gráficos. Observa cómo cada parte responde a un concepto visto antes.

**Componentes nuevos:**
- `st.sidebar`: coloca filtros en un panel lateral.
- `st.columns(n)`: organiza indicadores en columnas.
- `st.metric(etiqueta, valor)`: destaca un resultado.
- `st.line_chart(datos)`: dibuja una serie temporal.
- `st.dataframe(datos)`: muestra una tabla exploratoria.
- `st.tabs([...])`: organiza varias vistas.

**Importante:** calcularemos el rendimiento y la volatilidad de cada activo sobre sus observaciones disponibles; las comparaciones entre activos se basan en fechas comunes. Si no hay suficientes datos, la aplicación lo indicará.

In [14]:
%%writefile app.py
import streamlit as st
import yfinance as yf
import pandas as pd
import numpy as np

st.set_page_config(page_title="Observatorio financiero", layout="wide")
st.title("Observatorio de portafolios")
st.write("Compara rendimientos, riesgo y comportamiento temporal de acciones.")

activos = {"Apple": "AAPL", "Microsoft": "MSFT", "JPMorgan": "JPM", "NVIDIA": "NVDA", "Alphabet": "GOOGL"}
with st.sidebar:
    st.header("Configurar análisis")
    empresas = st.multiselect("Activos", list(activos), default=["Apple", "Microsoft"])
    periodo = st.selectbox("Período", ["3mo", "6mo", "1y", "2y"], index=2)
    ventana = st.selectbox("Media móvil (sesiones)", [20, 50], index=0)

@st.cache_data(ttl=3600, show_spinner=False)
def consultar_precios(ticker, periodo):
    return yf.Ticker(ticker).history(period=periodo, auto_adjust=True)

series = {}
for empresa in empresas:
    try:
        historial = consultar_precios(activos[empresa], periodo)
        if not historial.empty and "Close" in historial:
            serie = historial["Close"].dropna()
            if len(serie) >= 2:
                series[empresa] = serie.rename(empresa)
            else:
                st.warning(f"{empresa}: historial insuficiente.")
        else:
            st.warning(f"{empresa}: sin datos disponibles.")
    except Exception:
        st.warning(f"{empresa}: error al consultar los datos.")

if not series:
    st.info("Selecciona activos con datos disponibles para comenzar.")
    st.stop()

precios = pd.concat(series.values(), axis=1).sort_index()
# Fechas comunes para una comparación consistente
comparables = precios.dropna(how="any")
if len(comparables) < 2:
    st.warning("No hay suficientes fechas comunes para comparar los activos.")
    st.stop()

rendimientos = comparables.pct_change().dropna()
resumen = pd.DataFrame({
    "Último cierre (USD)": comparables.iloc[-1],
    "Rendimiento período (%)": (comparables.iloc[-1] / comparables.iloc[0] - 1) * 100,
    "Volatilidad anualizada (%)": rendimientos.std() * np.sqrt(252) * 100
}).round(2)

st.subheader("Indicadores por activo")
for empresa in resumen.index:
    a, b, c = st.columns(3)
    a.metric(f"{empresa} · Último cierre", f"USD {resumen.loc[empresa, 'Último cierre (USD)']:,.2f}")
    b.metric("Rendimiento del período", f"{resumen.loc[empresa, 'Rendimiento período (%)']:.2f}%")
    vol = resumen.loc[empresa, "Volatilidad anualizada (%)"]
    c.metric("Volatilidad anualizada", f"{vol:.2f}%" if pd.notna(vol) else "N/D")

p1, p2, p3, p4 = st.tabs(["Precios", "Rendimientos", "Tendencia", "Correlación"])
with p1:
    st.write("Precios de cierre en USD. Compara niveles, no rentabilidades.")
    st.line_chart(comparables)
with p2:
    st.write("Evolución de 100 unidades monetarias invertidas al inicio (sin costos ni dividendos en efectivo separados).")
    base100 = comparables.div(comparables.iloc[0]).mul(100)
    st.line_chart(base100)
    st.write("Rendimientos diarios (%)")
    st.line_chart(rendimientos * 100)
with p3:
    st.write(f"Media móvil de {ventana} sesiones bursátiles.")
    st.line_chart(comparables.rolling(ventana).mean())
with p4:
    if len(series) > 1 and len(rendimientos) >= 2:
        st.write("Correlación de rendimientos diarios")
        st.dataframe(rendimientos.corr().round(3))
    else:
        st.info("Selecciona al menos dos activos y un período con suficientes datos.")

st.subheader("Resumen comparativo")
st.dataframe(resumen)
st.caption("Fuente: Yahoo Finance vía yfinance. Resultados históricos, sujetos a disponibilidad y ajustes de datos; no son recomendaciones de inversión.")

Overwriting app.py


## 10 · Leer el tablero como analista financiero
Contesta con base en los datos que aparecen **en tu propia aplicación**:

1. ¿Cuál de los activos seleccionados tuvo mayor rendimiento acumulado? Indica el período analizado.
2. ¿Cuál presentó mayor volatilidad anualizada? ¿Cómo interpretas esa medida?
3. ¿La clasificación por rendimiento cambia cuando seleccionas otro horizonte? Presenta evidencia.
4. ¿Qué aporta el gráfico de base 100 frente al gráfico de precios de cierre?
5. ¿Qué puedes observar al incorporar la media móvil? ¿Qué información se pierde al suavizar la serie?
6. Si elegiste dos activos, ¿qué indica su correlación? ¿Qué **no** permite concluir?

**Reto de programación:** incorpora un activo nuevo al diccionario, añade un texto interpretativo dentro de la aplicación y comprueba que los controles siguen funcionando.

## 11 · Guardar y entregar tu trabajo
**Evidencia individual en GitHub:**
- `app.py`: código de la aplicación.
- `requirements.txt`: dependencias de Python.
- `README.md`: propósito, instrucciones para ejecutar en Colab y descripción de indicadores.
- Notebook de Colab: explicación del proceso y respuestas a las preguntas de interpretación.
- Captura de la aplicación funcionando (sin datos privados).

**Recuerda:** el enlace temporal de LocalTunnel puede caducar; no es el entregable principal. Se evalúan el código, la comprensión y el análisis.

**Descarga los archivos de la aplicación desde Colab:**

In [ ]:
with open("requirements.txt", "w") as archivo:
    archivo.write("streamlit
yfinance
pandas
numpy
")
from google.colab import files
files.download("app.py")
files.download("requirements.txt")

## 12 · Conexión con R: ¿cómo sería esta aplicación en Shiny?
En Python utilizamos **Streamlit**. En R podemos utilizar **Shiny** para crear aplicaciones interactivas. Ambos separan conceptualmente la entrada del usuario, el procesamiento y la presentación de resultados.

| En Streamlit (Python) | En Shiny (R) | Para qué sirve |
|---|---|---|
| `st.title()` | `titlePanel()` | Título de la aplicación |
| `st.multiselect()` | `selectInput(..., multiple=TRUE)` | Seleccionar activos |
| `st.metric()` | `valueBox()` (según paquete) o `textOutput()` | Mostrar indicadores |
| `st.line_chart()` | `plotOutput()` + `renderPlot()` | Visualizar series |
| Cambio de widget y nueva ejecución del script | Reactividad con `reactive()` y `render*()` | Actualizar resultados |

En RStudio, la aplicación Shiny se ejecuta con `shiny::runApp()`; **no se ejecuta dentro de este notebook Python de Colab**. La práctica equivalente en R debe explicar por separado `ui`, `server`, los datos y los cálculos.

**Reflexión:** ¿qué conceptos se mantienen aunque cambie el lenguaje de programación?

## Recursos de consulta
- [Documentación oficial de Streamlit](https://docs.streamlit.io/)
- [Componentes de Streamlit](https://docs.streamlit.io/develop/api-reference)
- [Documentación de yfinance](https://ranaroussi.github.io/yfinance/)
- [Documentación de Pandas](https://pandas.pydata.org/docs/)
- [Introducción oficial a Shiny para R](https://shiny.posit.co/r/getstarted/)

**Antes de la clase:** la docente debe comprobar que Colab permite iniciar el servidor y que el servicio de túnel funciona en la red institucional.

## Continuación en RStudio
Streamlit funciona con Python. Para reproducir la experiencia en R se usa **Shiny**, que separa la interfaz (`ui`) y el procesamiento reactivo (`server`). Consulta el apartado de Shiny en la guía HTML y el archivo `app_financiera_shiny.R` como referencia.